# Monitor de un run

Para mirar cómo va un entrenamiento **mientras corre** en otro notebook, sin interrumpirlo. Ábrelo en otra pestaña con un entorno de **CPU**, que casi no gasta unidades: solo lee lo que el entrenamiento va guardando en Drive.

Lo que se ve aquí puede venir con unos minutos de atraso, lo que Drive tarde en sincronizar entre las dos sesiones. Si parece que no avanza, vuelve a correr la celda 1.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → **CPU**.

## 1. Montar Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

## 2. Traer el código

Solo hace falta clonarlo: `progress.py` no necesita instalar nada.

In [ ]:
%%bash
if [ -d /content/r2dreamer-cookie ]; then
  cd /content/r2dreamer-cookie && git pull --ff-only
else
  git clone https://github.com/FabriRandon/r2dreamer-cookie.git /content/r2dreamer-cookie
fi

## 3. Elegir el run

In [ ]:
LOGDIR = "/content/drive/MyDrive/galleta/semana8/cookie_full_s0_base_nano"

## 4. Ver el progreso

Córrela cuando quieras. Dice en qué paso va y cómo vienen las últimas partidas comparadas con un agente al azar, y dibuja las curvas: galletas, veces que aprieta el botón, casillas visitadas, pasos en el cuarto del botón, la recompensa que predice el modelo y la entropía del actor. La línea gris es un agente al azar y la verde punteada, el paso en que el run de Nano empezó a aprender.

Cómo leerlo, según el run de Nano:

- **Antes de ~600.000 pasos** lo normal es no ver galletas. Lo que sí debería verse es que el agente recorre el mapa (casillas visitadas cerca del azar o más) y que va al cuarto del botón y lo aprieta.
- **Entre ~600.000 y ~800.000** deberían aparecer galletas.
- **Si a los ~800.000 sigue en 0**, conviene cortar: a esa altura Nano ya sacaba ~20 por partida.

In [ ]:
!cd /content/r2dreamer-cookie && python3 progress.py --logdir "{LOGDIR}" --out /content/progreso.png

from IPython.display import Image, display

display(Image("/content/progreso.png"))

## 5. (Opcional) Ver al agente jugar

Necesita instalar el entorno (unos minutos) y corre en CPU. Usa el último checkpoint guardado, así que muestra al agente tal como iba en ese momento.

In [ ]:
%%bash
pip install -q uv
cd /content/r2dreamer-cookie
uv venv --python 3.11 .venv
uv pip install --python .venv/bin/python -e ".[cookie]" 2>&1 | tail -3

In [ ]:
SEMILLA_EPISODIO = 0
VIDEO = f"{LOGDIR}/agente_episodio{SEMILLA_EPISODIO}.mp4"

!cd /content/r2dreamer-cookie && .venv/bin/python watch.py --logdir "{LOGDIR}" --out "{VIDEO}" \
    --seed {SEMILLA_EPISODIO} 2>&1 | tail -n 1

from IPython.display import Video
Video(VIDEO, embed=True, width=760)